In [ ]:
import os

os.getcwd()
os.path.abspath('words.txt')


'/Users/khoivo/python-training/python-training/think_python'

In [14]:
num_years = 1.5
num_camels = 23

writer = open('output.txt', 'w')
writer.write(f'Years of observation: {num_years}\n')
writer.write(f'Camels spotted: {num_camels}\n')
writer.close()

open('output.txt').read()

'Years of observation: 1.5\nCamels spotted: 23\n'

In [ ]:
import yaml

config = {
    'photo_dir': 'photos',
    'data_dir': 'photo_info',
    'extensions': ['jpg', 'jpeg'],
}

config_filename = 'config.yaml'
writer = open(config_filename, 'w')
yaml.dump(config, writer)
writer.close()

readback = open(config_filename).read()
print(readback)

reader = open(config_filename)
config_readback = yaml.safe_load(reader)
config_readback



data_dir: photo_info
extensions:
- jpg
- jpeg
photo_dir: photos



{'data_dir': 'photo_info',
 'extensions': ['jpg', 'jpeg'],
 'photo_dir': 'photos'}

In [24]:
os.makedirs(config['data_dir'], exist_ok=True)

db_file = os.path.join(config['data_dir'], 'captions')
db_file

import shelve

db = shelve.open(db_file, 'c')
db

for key in db:
    print(key, ':', db[key])

In [27]:
def sort_word(word):
    return ''.join(sorted(word))
  
word = 'pots'
key = sort_word(word)
key

db = shelve.open('anagram_map', 'n')
db[key] = [word]
db[key]

['pots']

In [ ]:
import hashlib
import os

config = {
    'photo_dir': 'photos',
    'data_dir': 'photo_info',
    'extensions': ['jpg', 'jpeg'],
}

path = os.getcwd()

photo_dir = os.path.join(path, config['photo_dir'])
os.makedirs(photo_dir, exist_ok=True)

path1 = os.path.join(photo_dir, 'photo1.jpg')
if not os.path.exists(path1):
    with open(path1, 'wb') as f:
        f.write(b'This is a placeholder for photo1.jpg')

data1 = open(path1, 'rb').read()
type(data1)

path2 = os.path.join(photo_dir, 'photo2.jpg')
if not os.path.exists(path2):
    with open(path2, 'wb') as f:
        f.write(b'This is a placeholder for photo2.jpg')

data2 = open(path2, 'rb').read()
data1 == data2

md5_hash = hashlib.md5()
type(md5_hash)

_hashlib.HASH

In [ ]:
def md5_digest(filename):
    data = open(filename, 'rb').read()
    md5_hash = hashlib.md5()
    md5_hash.update(data)
    digest = md5_hash.hexdigest()
    return digest

In [9]:
def replace_all(old, new, source_path, dest_path):
  with open(source_path, 'r') as reader:
    contents = reader.read()
  
  updated_contents = contents.replace(old, new)
  
  with open(dest_path, 'w') as writer:
    writer.write(updated_contents)

source_file = os.path.join(photo_dir, 'notes.txt')
dest_file = os.path.join(photo_dir, 'new_notes.txt')

if not os.path.exists(source_file):
  with open(source_file, 'w') as f:
    f.write("This directory contains photos of various events.")

replace_all('photos', 'images', source_file, dest_file)

with open(dest_file, 'r') as f:
  print(f.read())
  
def add_word(word, shelf):
  key = ''.join(sorted(word))
  if key in shelf:
    shelf[key].append(word)
  else:
    shelf[key] = [word]
  shelf.sync()  

This directory contains images of various events.


In [13]:
import shelve

def md5_digest(filename):
    data = open(filename, 'rb').read()
    md5_hash = hashlib.md5()
    md5_hash.update(data)
    digest = md5_hash.hexdigest()
    return digest
  
def is_image(path, extensions):
  """Check if the file is an image based on its extension."""
  _, ext = os.path.splitext(path)
  return ext.lower() in (f".{ext}" for ext in extensions)

def add_path(path, shelf):
  """Add a file path to the shelf based on its MD5 digest."""
  digest = md5_digest(path)
  if digest in shelf:
    shelf[digest].append(path)
  else:
    shelf[digest] = [path]
  shelf.sync()

def walk_images(directory, extensions, shelf):
  """Walk through a directory and add image files to the shelf."""
  for root, _, files in os.walk(directory):
    for file in files:
      path = os.path.join(root, file)
      if is_image(path, extensions):
        add_path(path, shelf)

def same_contents(path1, path2):
  """Check if two files have the same contents."""
  return open(path1, 'rb').read() == open(path2, 'rb').read()

# Ensure shelve is used correctly
db = shelve.open(os.path.join(config['photo_dir'], 'digests'), 'n')
walk_images(config['photo_dir'], config['extensions'], db)

for digest, paths in db.items():
  if len(paths) > 1:
    print(paths)
    if same_contents(paths[0], paths[1]):
      print(f"Files {paths[0]} and {paths[1]} have the same contents.")